In [ ]:
# V1 — Projectile Motion Verification
# Tennis Serve Admissibility Project

import numpy as np
import matplotlib.pyplot as plt

G = 9.81  # gravitational acceleration (m/s^2)

print("V1 environment ready.")
print(f"Gravity: {G} m/s²")


In [ ]:
# Cell 2 — Projectile Dynamics and RK4 Solver

def analytical_trajectory(t, v0, theta, x0=0.0, z0=3.0):
    """Exact 2D projectile trajectory under gravity only."""
    vx0 = v0 * np.cos(theta)
    vz0 = v0 * np.sin(theta)

    x = x0 + vx0 * t
    z = z0 + vz0 * t - 0.5 * G * t**2

    return x, z


def projectile_derivative(t, state, g=G):
    """Equations of motion for a gravity-only projectile.

    State vector: [x, z, vx, vz]
    """
    x, z, vx, vz = state

    return np.array([
        vx,       # dx/dt
        vz,       # dz/dt
        0.0,      # dvx/dt
        -g        # dvz/dt
    ])


def rk4_step(f, t, y, dt, **kwargs):
    """Perform one fourth-order Runge-Kutta integration step."""
    k1 = np.asarray(f(t, y, **kwargs), dtype=float)
    k2 = np.asarray(f(t + dt / 2, y + dt * k1 / 2, **kwargs), dtype=float)
    k3 = np.asarray(f(t + dt / 2, y + dt * k2 / 2, **kwargs), dtype=float)
    k4 = np.asarray(f(t + dt, y + dt * k3, **kwargs), dtype=float)

    return y + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)


def rk4_trajectory(v0, theta, z0=3.0, t_final=1.0, dt=0.01):
    """Compute a gravity-only projectile trajectory using RK4.

    State vector: [x, z, vx, vz]
    """
    n_steps = int(round(t_final / dt))
    times = np.linspace(0.0, t_final, n_steps + 1)
    dt_actual = times[1] - times[0] if n_steps > 0 else t_final

    vx0 = v0 * np.cos(theta)
    vz0 = v0 * np.sin(theta)

    state = np.array([0.0, z0, vx0, vz0], dtype=float)
    states = np.zeros((len(times), 4))
    states[0] = state

    for i in range(1, len(times)):
        state = rk4_step(
            projectile_derivative,
            times[i - 1],
            state,
            dt_actual,
            g=G
        )
        states[i] = state

    return times, states


# Example projectile parameters
SPEED_KMH = 200.0
v0 = SPEED_KMH / 3.6          # km/h → m/s
theta = np.radians(6.0)       # 6° launch angle
z0 = 3.0                      # 3 m contact height

print(f"Initial speed: {SPEED_KMH:.1f} km/h = {v0:.6f} m/s")
print(f"Launch angle: {np.degrees(theta):.1f}°")
print(f"Contact height: {z0:.2f} m")

# Run RK4 for the verification projectile
t_rk4, states_rk4 = rk4_trajectory(
    v0=v0,
    theta=theta,
    z0=z0,
    t_final=1.0,
    dt=0.01
)

x_rk4 = states_rk4[:, 0]
z_rk4 = states_rk4[:, 1]

print(f"RK4 steps: {len(t_rk4) - 1}")
print(f"Final x: {x_rk4[-1]:.6f} m")
print(f"Final z: {z_rk4[-1]:.6f} m")


In [ ]:
# Cell 3 — Analytical Solution vs RK4

x_exact, z_exact = analytical_trajectory(
    t_rk4,
    v0,
    theta,
    z0=z0
)

x_error = np.abs(x_rk4 - x_exact)
z_error = np.abs(z_rk4 - z_exact)

max_x_error = np.max(x_error)
max_z_error = np.max(z_error)

print(f"Maximum x-position error: {max_x_error:.12e} m")
print(f"Maximum z-position error: {max_z_error:.12e} m")


In [ ]:
# Cell 4 — Analytical vs RK4 Plot

plt.figure(figsize=(10, 5))

plt.plot(x_exact, z_exact, label="Analytical solution")
plt.plot(x_rk4, z_rk4, "--", label="RK4")

plt.xlabel("Horizontal position x (m)")
plt.ylabel("Height z (m)")
plt.title("Analytical Projectile vs RK4")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Cell 5 — RK4 Convergence Verification
#
# A nonlinear benchmark with a known analytical solution is used
# to verify the expected fourth-order convergence of RK4.

def nonlinear_derivative(t, y):
    """Nonlinear test equation: dy/dt = -y^2."""
    return -y**2


def rk4_scalar(f, y0, t_final, dt):
    """RK4 solver for a scalar differential equation."""
    n_steps = int(round(t_final / dt))
    times = np.linspace(0.0, t_final, n_steps + 1)
    dt_actual = times[1] - times[0] if n_steps > 0 else t_final

    y = np.zeros(len(times))
    y[0] = y0

    for i in range(1, len(times)):
        t = times[i - 1]
        yi = y[i - 1]

        k1 = f(t, yi)
        k2 = f(t + dt_actual / 2, yi + dt_actual * k1 / 2)
        k3 = f(t + dt_actual / 2, yi + dt_actual * k2 / 2)
        k4 = f(t + dt_actual, yi + dt_actual * k3)

        y[i] = yi + (dt_actual / 6) * (k1 + 2*k2 + 2*k3 + k4)

    return times, y


dt_values = np.array([0.20, 0.10, 0.05, 0.025, 0.0125])

# Confirm the convergence study uses exact factor-of-two refinement.
ratios = dt_values[:-1] / dt_values[1:]
assert np.allclose(ratios, 2.0), (
    "Convergence study requires a factor-of-two timestep refinement."
)

errors = []

for dt in dt_values:
    t_test, y_rk4 = rk4_scalar(
        nonlinear_derivative,
        y0=1.0,
        t_final=1.0,
        dt=dt
    )

    y_exact = 1 / (1 + t_test)
    error = np.max(np.abs(y_rk4 - y_exact))
    errors.append(error)

errors = np.array(errors)

print("RK4 convergence study")
print("-" * 50)
for dt, error in zip(dt_values, errors):
    print(f"dt = {dt:7.4f} s   error = {error:.8e}")


In [ ]:
# Cell 6 — Observed Convergence Order

orders = []

for i in range(len(errors) - 1):
    E1 = errors[i]
    E2 = errors[i + 1]
    dt1 = dt_values[i]
    dt2 = dt_values[i + 1]

    p = np.log(E1 / E2) / np.log(dt1 / dt2)
    orders.append(p)

orders = np.array(orders)

print("Observed RK4 convergence order")
print("-" * 50)
for i, p in enumerate(orders):
    print(f"{dt_values[i]:.4f} → {dt_values[i+1]:.4f} s   p = {p:.6f}")

print()
print(f"Average observed order: {np.mean(orders):.6f}")
print(f"Expected RK4 order:     4.000000")


In [ ]:
# Cell 7 — RK4 Convergence Plot

plt.figure(figsize=(8, 5))

plt.loglog(dt_values, errors, "o-", label="RK4 error")

reference = errors[-1] * (dt_values / dt_values[-1])**4
plt.loglog(dt_values, reference, "--", label="4th-order reference")

plt.xlabel("Time step Δt (s)")
plt.ylabel("Maximum absolute error")
plt.title("RK4 Convergence Study")
plt.legend()
plt.grid(True, which="both")
plt.show()


In [ ]:
# Cell 8 — Automated RK4 Verification

finest_order = orders[-1]
mean_order_error = np.mean(np.abs(orders - 4.0))

projectile_pass = (
    max_x_error < 1e-10 and
    max_z_error < 1e-10
)

order_pass = (
    abs(finest_order - 4.0) < 0.05 and
    mean_order_error < 0.10
)

convergence_grid_pass = np.allclose(
    dt_values[:-1] / dt_values[1:],
    2.0
)

print("=" * 55)
print("V1 NUMERICAL VERIFICATION")
print("=" * 55)
print(f"Projectile x-error:       {max_x_error:.3e} m")
print(f"Projectile z-error:       {max_z_error:.3e} m")
print(f"Finest observed order:    {finest_order:.6f}")
print(f"Mean |p - 4|:             {mean_order_error:.6f}")
print(f"Timestep ratio check:     {convergence_grid_pass}")
print(f"Expected RK4 order:       4.000000")
print()

print("✓ Analytical projectile verification: " + ("PASS" if projectile_pass else "FAIL"))
print("✓ Fourth-order convergence verification: " + ("PASS" if order_pass else "FAIL"))
print("✓ Timestep refinement verification: " + ("PASS" if convergence_grid_pass else "FAIL"))

overall_pass = projectile_pass and order_pass and convergence_grid_pass
print()
print("V1 VERIFICATION: " + ("PASS" if overall_pass else "REVIEW REQUIRED"))
assert overall_pass, "V1 numerical verification failed."


## V1 — Numerical Verification

The gravity-only projectile model was used as the first numerical verification problem for the tennis-serve trajectory framework.

Two independent checks were performed:

1. **Analytical agreement:** The RK4 trajectory was compared with the exact projectile solution. The maximum position error should remain at approximately floating-point precision for this constant-acceleration system.

2. **Convergence verification:** Because the gravity-only equations are integrated essentially exactly by RK4, their roundoff-level error is not useful for measuring convergence order. An independent nonlinear benchmark,

$$
\frac{dy}{dt}=-y^2, \qquad y(0)=1,
$$

with exact solution

$$
y(t)=\frac{1}{1+t},
$$

was therefore used to verify the expected fourth-order convergence. The timestep sequence is refined by a factor of two at every step.

**Conclusion:** V1 provides numerical verification of the RK4 implementation before aerodynamic forces are introduced.
